## Batch processing API to get path explanation

In [29]:
import anthropic
import json
import csv
import time
from pathlib import Path
import pandas as pd

client = anthropic.Anthropic()



In [46]:
def prepare_batch_requests(csv_file_path, output_jsonl_path, start_index=0, end_index=300):
    """
    Convert CSV data into batch API JSONL format.
    Each line is a separate request for KG triplet extraction.
    
    Args:
        csv_file_path: Path to input CSV file
        output_jsonl_path: Path to output JSONL file
        start_index: Starting row index (0-based, inclusive)
        end_index: Ending row index (0-based, exclusive)
    """
    with open(csv_file_path, 'r', encoding='utf-8') as csv_file, \
         open(output_jsonl_path, 'w', encoding='utf-8') as jsonl_file:
        
        reader = csv.DictReader(csv_file)
        
        for idx, row in enumerate(reader):
            # Skip rows before start_index
            if idx < start_index:
                continue
            # Stop after end_index
            if idx >= end_index:
                break
                
            text_id = row['id']  # Adjust column name as needed
            text_triplets = row['document_triplets']  # Adjust column name as needed
            summary_triplets = row['gen_summary_triplets']  # Adjust column name as needed
            
            # Create the batch request
            batch_request = {
                "custom_id": f"kg-extraction-{text_id}",
                "params": {
                    "model": "claude-sonnet-4-20250514",
                    "max_tokens": 20000,
                    "system": "You are a helpful assistant in traversing and reasoning about knowledge graphs",
                    "messages": [
                        {
                            "role": "user",
                            "content": f"""
        You are given a set of Text Triplets extracted from a source document and a set of Summary Triplets extracted from a generated summary. Your task is to determine whether each Summary Triplet is supported by the Text Triplets, and to identify the explicit evidence path that justifies it.

Follow the procedure strictly.

### Procedure

1. **Knowledge Graph Construction**
   Construct a knowledge graph using only the provided Text Triplets.Each triplet corresponds to a directed edge in the graph.

2. **Evidence Path Search**
   For each Summary Triplet:
   - Attempt to derive the Summary Triplet by traversing the knowledge graph.
   - Use single-hop or multi-hop traversal only when each hop corresponds to an explicit Text Triplet.
   - A valid multi-hop path must form a connected chain from the subject entity to the object entity.

3. **Strict Grounding Constraint**
   - Do not introduce external world knowledge, common-sense assumptions, or narrative inference.
   - Do not resolve entities (e.g., “the city”, jurisdictions, roles) unless explicitly stated in the Text Triplets.
   - If any required step is not grounded in the Text Triplets, the Summary Triplet is not supported.

4. **Unsupported Cases**
   - If no valid evidence path exists, identify the closest relevant Text Triplets and explain why they fail to support the Summary Triplet.

### Output Format

Return only a Python list with the following structure:
<python>
[
{{
"summary_triple": ["entity1", "relation", "entity2"],
"is_supported": true | false,
"supporting_evidence":
[list of Text Triplets forming a connected evidence path]
OR
[list of relevant but insufficient Text Triplets if unsupported],
"confidence": 0-2 (include only if supported; None otherwise),
"reasoning": "Explain the explicit graph-based derivation OR clearly state which link is missing"
}}
]
</python>

### Guidelines

- Mark a Summary Triplet as supported only if it can be derived entirely from the Text Triplets via explicit graph traversal.
- Semantic equivalence (e.g., synonyms) is allowed only if it does not introduce new relations.
- Multi-hop reasoning is valid only when every hop is explicitly grounded in the Text Triplets.
- Be conservative: if support is ambiguous or incomplete, mark the Summary Triplet as unsupported.

Text triplets = {text_triplets}
Summary Triplets = {summary_triplets}

    """
                        }
                    ]
                }
            }
            
            jsonl_file.write(json.dumps(batch_request) + '\n')
    
    print(f"Batch requests prepared and saved to {output_jsonl_path}")

In [47]:
def submit_batch(jsonl_file_path):
    """
    Submit the batch of requests to Claude API.
    Returns the batch ID for tracking.
    """
    # Read all requests from the JSONL file
    requests = []
    with open(jsonl_file_path, 'r', encoding='utf-8') as f:
        for line in f:
            requests.append(json.loads(line))
    
    # Create the message batch
    message_batch = client.messages.batches.create(
        requests=requests
    )
    
    print(f"Batch submitted successfully!")
    print(f"Batch ID: {message_batch.id}")
    print(f"Status: {message_batch.processing_status}")
    
    return message_batch.id

def check_batch_status(batch_id):
    """
    Check the status of a batch processing job.
    """
    batch = client.messages.batches.retrieve(batch_id)
    
    print(f"\nBatch Status: {batch.processing_status}")
    print(f"Requests: {batch.request_counts}")
    
    return batch.processing_status

def retrieve_batch_results(batch_id, output_jsonl_path):
    """
    Retrieve results once batch processing is complete.
    """
    # Get all results
    results = client.messages.batches.results(batch_id)
    
    with open(output_jsonl_path, 'w', encoding='utf-8') as f:
        for result in results:
            f.write(json.dumps(result.model_dump()) + '\n')
    
    print(f"Results saved to {output_jsonl_path}")

def process_results(results_jsonl_path, output_csv_path):
    """
    Process the batch results and extract KG triplets into a CSV.
    """
    triplets = []
    
    with open(results_jsonl_path, 'r', encoding='utf-8') as f:
        for line in f:
            result = json.loads(line)
            
            # Extract custom_id to get original text ID
            custom_id = result['custom_id']
            text_id = custom_id.replace('kg-extraction-', '')
            
            # Check if request was successful
            if result['result']['type'] == 'succeeded':
                response = result['result']['message']
                
                # Extract the triplets from the response
                content = response['content'][0]['text']
                
                try:
                    # Parse the JSON response
                    extracted_triplets = json.loads(content)
                    
                    # Add text_id to each triplet
                    for triplet in extracted_triplets:
                        triplets.append({
                            'text_id': text_id,
                            'subject': triplet['subject'],
                            'predicate': triplet['predicate'],
                            'object': triplet['object']
                        })
                except json.JSONDecodeError:
                    print(f"Warning: Could not parse response for {custom_id}")
            else:
                print(f"Error processing {custom_id}: {result['result']}")
    
    # Write to CSV
    with open(output_csv_path, 'w', newline='', encoding='utf-8') as csvfile:
        fieldnames = ['text_id', 'subject', 'predicate', 'object']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        
        writer.writeheader()
        writer.writerows(triplets)
    
    print(f"Processed {len(triplets)} triplets and saved to {output_csv_path}")


In [48]:
def main():
    """
    Main workflow for batch KG extraction.
    """
    #     # File paths
    input_csv = "../data/train/data_with_triplets.csv"
    batch_requests_file = "../data/test/batch_requests_traversal_v2.jsonl"
    batch_results_file = "../data/test/batch_results_traversal_v2.jsonl"
    output_csv = "../data/test/kg_triplets_traversal.csv"
    
    # Step 1: Prepare batch requests
    print("Step 1: Preparing batch requests...")
    prepare_batch_requests(input_csv, batch_requests_file)
    
    # Step 2: Submit batch
    print("\nStep 2: Submitting batch...")
    batch_id = submit_batch(batch_requests_file)
    
    # Step 3: Poll for completion
    print("\nStep 3: Waiting for batch to complete...")
    while True:
        status = check_batch_status(batch_id)
        
        if status == "ended":
            break
        
        print("Still processing... checking again in 60 seconds")
        time.sleep(60)
    
    # Step 4: Retrieve results
    print("\nStep 4: Retrieving results...")
    retrieve_batch_results(batch_id, batch_results_file)



In [49]:
if __name__ == "__main__":
    main()

Step 1: Preparing batch requests...
Batch requests prepared and saved to ../data/test/batch_requests_traversal_v2.jsonl

Step 2: Submitting batch...
Batch submitted successfully!
Batch ID: msgbatch_01Ac8osRDMFeeVHBrP6evbfT
Status: in_progress

Step 3: Waiting for batch to complete...
Batch submitted successfully!
Batch ID: msgbatch_01Ac8osRDMFeeVHBrP6evbfT
Status: in_progress

Step 3: Waiting for batch to complete...

Batch Status: in_progress
Requests: MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=300, succeeded=0)
Still processing... checking again in 60 seconds

Batch Status: in_progress
Requests: MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=300, succeeded=0)
Still processing... checking again in 60 seconds

Batch Status: in_progress
Requests: MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=300, succeeded=0)
Still processing... checking again in 60 seconds

Batch Status: in_progress
Requests: MessageBatchReque

### Format retrived output

In [50]:
def extract_python_list_from_text(text_content):
    """
    Extract and parse a Python list from text content that contains markdown code blocks.
    
    Args:
        text_content: String containing text with a Python list in a code block
        
    Returns:
        Parsed Python list object, or None if extraction fails
    """
    import re
    import ast
    
    # Find content between <python> and </python>
    pattern = r'<python>\s*(.*?)\s*</python>'
    match = re.search(pattern, text_content, re.DOTALL)
    
    if match:
        list_str = match.group(1).strip()
        
        # Method 1: Try JSON parsing first (handles true/false/null)
        try:
            parsed_list = json.loads(list_str)
            return parsed_list
        except json.JSONDecodeError as e:
            # JSON failed, likely due to formatting issues
            pass
        
        # Method 2: Try ast.literal_eval (handles Python literals like True/False/None)
        try:
            # Replace JSON booleans/null with Python equivalents
            python_str = list_str.replace('true', 'True').replace('false', 'False').replace('null', 'None')
            parsed_list = ast.literal_eval(python_str)
            return parsed_list
        except (ValueError, SyntaxError) as e:
            # ast.literal_eval failed
            pass
        
        # Method 3: Manual JSON repair - fix common issues
        try:
            # Try to fix common JSON issues:
            # 1. Remove any trailing commas before closing braces/brackets
            fixed_str = re.sub(r',\s*([}\]])', r'\1', list_str)
            # 2. Ensure proper quotes (sometimes model uses smart quotes)
            fixed_str = fixed_str.replace('"', '"').replace('"', '"').replace("'", "'").replace("'", "'")
            
            parsed_list = json.loads(fixed_str)
            return parsed_list
        except json.JSONDecodeError:
            pass
        
        # Method 4: Last resort - use eval (unsafe but controlled environment)
        try:
            # Make true/false/null available for eval
            parsed_list = eval(list_str, {"__builtins__": {}, "true": True, "false": False, "null": None})
            return parsed_list
        except Exception as e:
            print(f"All parsing methods failed. Error: {e}")
            print(f"Content preview: {list_str[:200]}...")
            return None
    else:
        print("No <python> code block found in text")
        return None

In [51]:
def parse_results(batch_results_file, data_csv):

    """
    Parse batch results and align with original data CSV.
    
    Args:
        batch_results_file: Path to batch results JSONL file
        data_csv: Path to original data CSV file
    """
    llm_results = []

    df = pd.read_csv(data_csv)
    
    # Parse results
    with open(batch_results_file, 'r', encoding='utf-8') as f:
        for idx, line in enumerate(f):
            result = json.loads(line)
            id = result['custom_id'][14:]

            text_content = result['result']['message']['content'][0]['text']
            # print(text_content)

            extracted_list = extract_python_list_from_text(text_content)

            print(f"Extracted List for Text ID {id}: {extracted_list}")

            text_id = df.iloc[idx]['id']
            llm_results.append({
                'id': text_id,  # Convert to int to match original CSV
                'fact_eval_res': extracted_list
            })

    return pd.DataFrame(llm_results)
            


In [52]:
batch_resulstts_file = "../data/train/batch_results_traversal_v2.jsonl"
data_csv_file = "../data/train/data_with_triplets.csv"

original_df = pd.read_csv(data_csv_file)

llm_results_df = parse_results(batch_resulstts_file, data_csv_file)

# merge with original data if needed
merged_df = pd.merge(original_df, llm_results_df, on='id', how='left')

Extracted List for Text ID 0: [{'summary_triple': ['Clean-up operations', 'continuing across', 'Scottish Borders'], 'is_supported': False, 'supporting_evidence': [['flood alert', 'remains in place across', 'Borders'], ['Hawick', 'has ongoing', 'repair work'], ['Alex Rowley', 'was in', 'Hawick']], 'confidence': None, 'reasoning': "While there is evidence of ongoing repair work in Hawick and flood alerts across Borders, there is no explicit Text Triplet stating that 'Clean-up operations' are 'continuing across' Scottish Borders. The closest evidence shows repair work and alerts, but does not directly establish clean-up operations."}, {'summary_triple': ['Clean-up operations', 'continuing across', 'Dumfries and Galloway'], 'is_supported': False, 'supporting_evidence': [['flooding', 'affected', 'Dumfries and Galloway'], ['Newton Stewart', 'worst affected by', 'damage'], ['Nicola Sturgeon', 'visited', 'Newton Stewart']], 'confidence': None, 'reasoning': "While Dumfries and Galloway was affe

In [53]:
display(llm_results_df.head())

,id,fact_eval_res
0,0,"[{'summary_triple': ['Clean-up operations', 'c..."
1,1,"[{'summary_triple': ['Two tourist buses', 'des..."
2,2,"[{'summary_triple': ['Lewis Hamilton', 'storme..."
3,3,[{'summary_triple': ['former Lincolnshire Poli...
4,4,"[{'summary_triple': ['armed man', 'locked hims..."


In [54]:
# save merged results if needed
merged_df.to_csv("../data/train/merged_results_traversal.csv", index=False)